# Notebook 3: Stationarity Testing + Ornstein-Uhlenbeck Fit

Validates that the HYSYS-weighted partial product spread is mean-reverting,
then fits an Ornstein-Uhlenbeck process to extract the parameters that drive
the trading signal.

## Train / test split

All parameter selection (stationarity validation, OU fit, z-score baseline)
is performed on the **training set only**. The **test set is never touched**
in this notebook or used to influence any parameter choice — it is held out
entirely for out-of-sample backtesting in Notebook 4.

```
|------------------- Train (70%) -------------------|------- Test (30%) -------|
        fit OU params, validate stationarity              held out completely
        choose z-score lookback/threshold                  used only in NB4
```

## Tests performed
1. **ADF (Augmented Dickey-Fuller)** — rejects unit root (H0: non-stationary)
2. **KPSS** — confirms stationarity (H0: stationary)
3. **OU parameter estimation** — Maximum Likelihood Estimation

## OU Process
```
dX(t) = theta*(mu - X(t))dt + sigma*dW(t)
```
- **theta:** mean reversion speed
- **mu:** long-run equilibrium spread
- **sigma:** volatility
- **Half-life:** ln(2)/theta - practical trading horizon in days

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from signal_generator import fit_ou_process, compute_zscore
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load spread data
df = pd.read_csv('../data/spread_data.csv', index_col=0, parse_dates=True)
df = df.dropna(subset=['spread_hysys'])
print(f'Total series length: {len(df)} observations')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT - chronological, 70/30
# Test set is held out completely from this point forward.
# No parameter in this notebook is chosen using test-set data.
# ============================================================

split_idx = int(len(df) * 0.70)
split_date = df.index[split_idx]

train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f'Train set: {len(train)} obs  ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Test set:  {len(test)} obs  ({test.index[0].date()} to {test.index[-1].date()})')
print(f'Split date: {split_date.date()}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index, train['spread_hysys'], color='#2c3e50', linewidth=0.8, label='Train (in-sample)')
ax.plot(test.index,  test['spread_hysys'],  color='#e67e22', linewidth=0.8, label='Test (held out)')
ax.axvline(split_date, color='red', linestyle='--', linewidth=1, label='Train/Test split')
ax.set_ylabel('Spread ($/bbl)')
ax.set_title('Train/Test Split - HYSYS Partial Product Spread')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STATIONARITY TESTS - TRAIN SET ONLY
# ============================================================

train_spread = train['spread_hysys']

# ADF Test - H0: unit root (non-stationary)
adf_stat, adf_pvalue, adf_lags, adf_nobs, adf_crit, _ = adfuller(train_spread, autolag='AIC')

print('=' * 55)
print('AUGMENTED DICKEY-FULLER TEST (TRAIN SET)')
print('H0: Unit root exists (series is NON-stationary)')
print('=' * 55)
print(f'ADF Statistic:  {adf_stat:.4f}')
print(f'p-value:        {adf_pvalue:.4f}')
print(f'Lags used:      {adf_lags}')
print('Critical values:')
for key, val in adf_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if adf_pvalue < 0.05:
    print('RESULT: Reject H0 at 5% significance -> series IS stationary (train set)')
else:
    print('RESULT: Fail to reject H0 -> series may NOT be stationary (train set)')

In [ ]:
# KPSS Test - TRAIN SET ONLY - H0: series IS stationary
kpss_stat, kpss_pvalue, kpss_lags, kpss_crit = kpss(train_spread, regression='c', nlags='auto')

print('=' * 55)
print('KPSS TEST (TRAIN SET)')
print('H0: Series IS stationary')
print('=' * 55)
print(f'KPSS Statistic: {kpss_stat:.4f}')
print(f'p-value:        {kpss_pvalue:.4f}')
print('Critical values:')
for key, val in kpss_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if kpss_pvalue > 0.05:
    print('RESULT: Fail to reject H0 -> series IS stationary (train set)')
else:
    print('RESULT: Reject H0 -> series may NOT be stationary (train set)')

In [ ]:
# ============================================================
# OU PARAMETER ESTIMATION (MLE) - TRAIN SET ONLY
# These parameters (theta, mu, sigma) are fixed after this cell
# and applied unchanged to the test set in Notebook 4.
# ============================================================

ou_params = fit_ou_process(train_spread)
theta     = ou_params['theta']
mu        = ou_params['mu']
sigma     = ou_params['sigma']
half_life = ou_params['half_life']

print('=' * 55)
print('ORNSTEIN-UHLENBECK PARAMETERS (MLE, TRAIN SET)')
print('dX = theta(mu - X)dt + sigma*dW')
print('=' * 55)
print(f'theta (mean reversion speed): {theta:.4f} per day')
print(f'mu (long-run mean):           ${mu:.4f}/bbl')
print(f'sigma (volatility):           ${sigma:.4f}/bbl/day^0.5')
print(f'Half-life:                    {half_life:.1f} trading days')
print(f'Optimisation converged:       {ou_params["converged"]}')
print()
print('These parameters are fixed from this point forward and applied')
print('unchanged to the held-out test set in Notebook 4 - no re-fitting on test data.')

In [ ]:
# ============================================================
# Z-SCORE - TRAIN SET (for threshold selection / visual inspection)
# The rolling lookback window (252d) is a mechanical calculation,
# not a fitted parameter - it can be applied identically to test
# data in Notebook 4 without constituting look-ahead bias, since
# at any point t it only uses data up to and including t.
# ============================================================

LOOKBACK = 252

train['margin_mean'] = train_spread.rolling(LOOKBACK).mean()
train['margin_std']  = train_spread.rolling(LOOKBACK).std()
train['zscore']      = (train_spread - train['margin_mean']) / train['margin_std']

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

axes[0].plot(train.index, train_spread, color='#2c3e50', linewidth=0.8, alpha=0.8, label='HYSYS spread (train)')
axes[0].plot(train.index, train['margin_mean'], color='#e74c3c', linewidth=1.5, label='Rolling mean (252d)')
axes[0].fill_between(train.index,
    train['margin_mean'] - train['margin_std'],
    train['margin_mean'] + train['margin_std'],
    alpha=0.15, color='#e74c3c', label='+/-1 sigma band')
axes[0].fill_between(train.index,
    train['margin_mean'] - 2*train['margin_std'],
    train['margin_mean'] + 2*train['margin_std'],
    alpha=0.08, color='#e74c3c', label='+/-2 sigma band')
axes[0].set_ylabel('Spread ($/bbl)')
axes[0].set_title(f'Train Set - HYSYS Spread - OU Half-life: {half_life:.0f} trading days')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(train.index, train['zscore'], color='#2c3e50', linewidth=0.8)
axes[1].axhline(0,    color='#e74c3c', linewidth=1, linestyle='-')
axes[1].axhline(1.5,  color='#e67e22', linewidth=1, linestyle='--', label='+/-1.5 sigma entry threshold')
axes[1].axhline(-1.5, color='#e67e22', linewidth=1, linestyle='--')
axes[1].fill_between(train.index, train['zscore'], 0,
    where=(train['zscore'] < -1.5), alpha=0.3, color='#27ae60', label='Long signal zone')
axes[1].fill_between(train.index, train['zscore'], 0,
    where=(train['zscore'] > 1.5),  alpha=0.3, color='#e74c3c', label='Short signal zone')
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/stationarity_ou_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# THRESHOLD SELECTION - TRAIN SET ONLY
# Sensitivity analysis to choose the entry threshold is done
# here on train data only. The chosen threshold is then applied,
# unchanged, to the test set in Notebook 4.
# ============================================================

from signal_generator import generate_signals, apply_holding_period

MIN_HOLDING_DAYS = 5
TRANSACTION_COST = 0.05

thresholds = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
results = []

margin_change_train = train_spread.diff()

for thresh in thresholds:
    sig  = generate_signals(train['zscore'], thresh, 0.0)
    sig  = apply_holding_period(sig, MIN_HOLDING_DAYS)
    pnl  = sig.shift(1) * margin_change_train - sig.diff().abs().clip(0,1) * TRANSACTION_COST
    sh   = (pnl.mean() / pnl.std()) * np.sqrt(252) if pnl.std() > 0 else 0
    ntrd = sig.diff().abs().clip(0,1).sum() / 2
    results.append({'threshold': thresh, 'sharpe': sh, 'num_trades': ntrd, 'total_pnl': pnl.sum()})

sensitivity = pd.DataFrame(results)
print('Threshold sensitivity (TRAIN SET ONLY):')
print(sensitivity.round(3))

best_threshold = sensitivity.loc[sensitivity['sharpe'].idxmax(), 'threshold']
print(f'\nSelected entry threshold (best train-set Sharpe): {best_threshold} sigma')
print('This threshold is now FIXED and will be applied to the held-out test set in Notebook 4.')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sensitivity['threshold'], sensitivity['sharpe'], 'o-', color='#2c3e50', linewidth=2)
ax.axvline(best_threshold, color='#e74c3c', linestyle='--', label=f'Selected: {best_threshold} sigma')
ax.set_xlabel('Entry Threshold (sigma)')
ax.set_ylabel('Train-set Sharpe Ratio')
ax.set_title('Threshold Selection - Train Set Only')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/threshold_selection_train.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save OU parameters, chosen threshold, and split info for notebook 4
params_to_save = pd.Series({
    'theta': theta,
    'mu': mu,
    'sigma': sigma,
    'half_life': half_life,
    'lookback': LOOKBACK,
    'entry_threshold': best_threshold,
    'split_idx': split_idx,
    'split_date': str(split_date.date())
})
params_to_save.to_csv('../data/ou_parameters.csv', header=['value'])

# Save full df with train/test flag for notebook 4
df['is_train'] = False
df.iloc[:split_idx, df.columns.get_loc('is_train')] = True
df.to_csv('../data/spread_full_with_split.csv')

print('Saved OU parameters, chosen threshold, and train/test split flags.')
print(params_to_save)